In [65]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report,f1_score

In [45]:
df=pd.read_csv("/content/archive (11).zip")
df

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


In [46]:
df.shape

(7043, 21)

In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [48]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [49]:
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [50]:
df = df.drop("customerID", axis=1)

In [51]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

In [52]:
df["Churn"]=df["Churn"].map({"Yes":1,"No":0})

In [53]:
df["TenureGroup"] = pd.cut(df["tenure"], bins=[-1, 12, 24, 48, 72],
                          labels=["0-12", "13-24", "25-48", "49-72"])

df["AvgMonthly"] = df["TotalCharges"] / (df["tenure"] + 1)
df["MonthlyToTotalRatio"] = df["MonthlyCharges"] / (df["AvgMonthly"] + 1e-5)

df["FiberOptic"] = (df["InternetService"] == "Fiber optic").astype(int)
df["NoInternet"] = (df["InternetService"] == "No").astype(int)

df["TotalServices"] = df[["PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
                         "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]].apply(
    lambda x: (x == "Yes").sum() if x.dtype == "object" else 0, axis=1)

df["ContractNum"] = df["Contract"].map({"Month-to-month": 0, "One year": 1, "Two year": 2})

df["SeniorPartner"] = ((df["SeniorCitizen"] == 1) & (df["Partner"] == "Yes")).astype(int)
df["NewCustomer"] = (df["tenure"] <= 6).astype(int)
df["HighCharge"] = (df["MonthlyCharges"] > df["MonthlyCharges"].quantile(0.75)).astype(int)

# ====================== One-hot Encoding ======================
df = pd.get_dummies(df, drop_first=True)

# Prepare X and y
X = df.drop("Churn", axis=1)
y = df["Churn"]

print("New shape after feature engineering:", X.shape)

New shape after feature engineering: (7043, 42)


In [54]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [55]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train, y_train = smote.fit_resample(X_train, y_train)

print(y_train.value_counts())

Churn
0    4139
1    4139
Name: count, dtype: int64


In [56]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [64]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [62]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

# Stacking Ensemble (usually stronger than Voting)
estimators = [
    ('lgb', LGBMClassifier(n_estimators=920,
    learning_rate=0.023,
    max_depth=6,
    num_leaves=43,
    min_child_samples=11,
    colsample_bytree=0.72,
    subsample=0.61,
    reg_alpha=0.39,
    reg_lambda=0.29,
    random_state=42)),   # use your best lgb params
    ('xgb', XGBClassifier(n_estimators=700, learning_rate=0.05, max_depth=7, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=600, max_depth=12, random_state=42))
]

stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=5,
    passthrough=True
)

stack.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


StackingClassifier(cv=5,
                   estimators=[('lgb',
                                LGBMClassifier(colsample_bytree=0.72,
                                               learning_rate=0.023, max_depth=6,
                                               min_child_samples=11,
                                               n_estimators=920, num_leaves=43,
                                               random_state=42, reg_alpha=0.39,
                                               reg_lambda=0.29,
                                               subsample=0.61)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_by...
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=7,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=700, n_jobs=None,
                                              num_parallel_tree=None, ...)),
                               ('rf',
                                RandomForestClassifier(max_depth=12,
                                                       n_estimators=600,
                                                       random_state=42))],
                   final_estimator=LogisticRegression(), passthrough=True)

In [63]:
probs = stack.predict_proba(X_test)[:, 1]

from sklearn.metrics import precision_recall_curve
import numpy as np

prec, rec, thresh = precision_recall_curve(y_test, probs)
f1_scores = 2 * prec * rec / (prec + rec + 1e-9)
best_idx = np.argmax(f1_scores)
best_thresh = thresh[best_idx]

pred = (probs > best_thresh).astype(int)

print(f"Best Threshold: {best_thresh:.3f}")
print(f"Best F1: {f1_scores[best_idx]:.4f}\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, pred))
print("\nClassification Report:")
print(classification_report(y_test, pred))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Best Threshold: 0.320
Best F1: 0.6303

Confusion Matrix:
[[781 254]
 [ 86 288]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.75      0.82      1035
           1       0.53      0.77      0.63       374

    accuracy                           0.76      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.76      0.77      1409



In [67]:
import joblib

joblib.dump(model, "churn_model.pkl")

print("Churn model saved")

Churn model saved


In [68]:
joblib.dump(scaler, "churn_scaler.pkl")

['churn_scaler.pkl']

In [69]:
import os

print(os.listdir())

['.config', 'catboost_info', 'churn_scaler.pkl', 'churn_model.pkl', 'archive (11).zip', 'sample_data']
